# LeetCode 3117 — Minimum Sum of Values by Dividing Array bằng Python

- Link: https://leetcode.com/problems/minimum-sum-of-values-by-dividing-array/
- Độ khó: **Hard**
- Chủ đề: **DP chia mảng thành `m` đoạn** với điều kiện AND của đoạn; với mỗi đầu phải `r`, các đầu trái `l` có `AND(l..r) = v` tạo thành **một đoạn liên tiếp** (AND đơn điệu) ⇒ chuyển trạng thái bằng **min trên đoạn** (sparse table) — `O(m · n log n)`; cách khác: DFS có nhớ `(i, j, AND hiện tại)`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minimumValueSum(self, nums: List[int], andValues: List[int]) -> int:
```

## 1. Mục tiêu học

1. "Giá trị" của một đoạn là **phần tử cuối** ⇒ tổng cần tối thiểu là tổng các phần tử ở vị trí cắt.
2. DP `dp_j[r]` = tổng nhỏ nhất khi chia `nums[0..r]` thành `j + 1` đoạn đúng `andValues[0..j]`.
3. Với `r` cố định, `AND(l..r)` không tăng khi `l` giảm và chỉ có `≤ 17` giá trị khác nhau ⇒ các `l` có `AND(l..r) = v` là một đoạn `[L, R]`.
4. `dp_j[r] = nums[r] + min_{l ∈ [L, R]} dp_{j−1}[l − 1]` ⇒ cần **min đoạn** trên mảng tầng trước — sparse table `O(1)` mỗi truy vấn.
5. Kiểm chứng bằng brute thử mọi cách cắt.

## 2. Đề bài & ví dụ

Cho `nums` (độ dài `n`) và `andValues` (độ dài `m`). Chia `nums` thành đúng `m` đoạn liên tiếp rời nhau sao cho AND của đoạn thứ `i` bằng `andValues[i]`. Giá trị của một đoạn là phần tử **cuối** của nó. Trả về tổng giá trị nhỏ nhất, hoặc `-1` nếu không chia được.

**Ví dụ 1:** `nums = [1,4,3,3,2]`, `andValues = [0,3,3,2]` → `12`.

**Ví dụ 2:** `nums = [2,3,5,7,7,7,5]`, `andValues = [0,7,5]` → `17`.

**Ví dụ 3:** `nums = [1,2,3,4]`, `andValues = [2]` → `-1`.

**Ràng buộc:** `1 ≤ n ≤ 10⁴`; `1 ≤ m ≤ min(n, 10)`; `1 ≤ nums[i] < 10⁵`; `0 ≤ andValues[j] < 10⁵`.

In [1]:
from typing import List
from functools import lru_cache
import itertools, random, time, sys

INF = float('inf')
examples = [([1, 4, 3, 3, 2], [0, 3, 3, 2], 12), ([2, 3, 5, 7, 7, 7, 5], [0, 7, 5], 17), ([1, 2, 3, 4], [2], -1)]
for a, v, exp in examples:
    print(a, v, '-> kỳ vọng', exp)

[1, 4, 3, 3, 2] [0, 3, 3, 2] -> kỳ vọng 12
[2, 3, 5, 7, 7, 7, 5] [0, 7, 5] -> kỳ vọng 17
[1, 2, 3, 4] [2] -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1. Nhóm các đầu trái theo giá trị AND

Với `r` cố định, khi `l` giảm từ `r` về 0, `AND(l..r)` chỉ tắt bớt bit ⇒ không tăng, đổi giá trị `≤ 17` lần (`nums < 2^17`). Ta duy trì danh sách **nhóm** `(giá trị, l_min, l_max)` cho các mảng con kết thúc tại `r`, cập nhật từ `r − 1` bằng cách AND thêm `nums[r]` và gộp nhóm trùng giá trị.

In [2]:
def groups_by_r(nums):
    out = []; cur = []                                   # cur: list (val, lmin, lmax), l giảm dần theo thứ tự
    for r, x in enumerate(nums):
        nxt = [(x, r, r)]
        for v, lo, hi in cur:
            w = v & x
            if w == nxt[-1][0]:
                nxt[-1] = (w, lo, nxt[-1][2])
            else:
                nxt.append((w, lo, hi))
        cur = nxt; out.append(cur)
    return out

g = groups_by_r(examples[1][0])
for r, grp in enumerate(g):
    print(f'r = {r}: (AND, l_min, l_max) =', grp)

r = 0: (AND, l_min, l_max) = [(2, 0, 0)]
r = 1: (AND, l_min, l_max) = [(3, 1, 1), (2, 0, 0)]
r = 2: (AND, l_min, l_max) = [(5, 2, 2), (1, 1, 1), (0, 0, 0)]
r = 3: (AND, l_min, l_max) = [(7, 3, 3), (5, 2, 2), (1, 1, 1), (0, 0, 0)]
r = 4: (AND, l_min, l_max) = [(7, 3, 4), (5, 2, 2), (1, 1, 1), (0, 0, 0)]
r = 5: (AND, l_min, l_max) = [(7, 3, 5), (5, 2, 2), (1, 1, 1), (0, 0, 0)]
r = 6: (AND, l_min, l_max) = [(5, 2, 6), (1, 1, 1), (0, 0, 0)]


### 3.2. Sparse table cho min đoạn

`sp[k][i] = min(a[i..i + 2^k − 1])`; truy vấn `[L, R]` lấy `min(sp[k][L], sp[k][R − 2^k + 1])` với `k = ⌊log₂(R − L + 1)⌋`. Dựng `O(n log n)`, hỏi `O(1)`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đi qua từng tầng `j` (đoạn thứ `j`). Muốn đoạn `j` kết thúc tại `r` với AND `= andValues[j]`: tra nhóm có giá trị đó trong danh sách nhóm của `r` ⇒ được khoảng đầu trái `[L, R]`; đoạn trước phải kết thúc tại `l − 1 ∈ [L − 1, R − 1]` ⇒ lấy `min` của tầng trước trên khoảng này.

Ví dụ 2: tầng cuối, `r = 6` (`nums[6] = 5`), `AND(l..6) = 5` khi `l ∈ [2..6]`? — `AND(2..6) = 5 & 7 & 7 & 7 & 5 = 5` ⇒ đầu trái hợp lệ `[3..6]` (vì `l ≥ 3` để đoạn trước còn phần tử)…; tầng trước tốt nhất kết thúc ở `5` (giá trị `5 + 7 = 12`) ⇒ `17`.

## 5. Cách 1 — Brute Force: thử mọi cách đặt `m − 1` vết cắt

In [3]:
def brute(nums, vals):
    n, m = len(nums), len(vals); best = INF
    for cuts in itertools.combinations(range(1, n), m - 1):
        b = [0] + list(cuts) + [n]; s = 0; ok = True
        for j in range(m):
            a = -1
            for x in nums[b[j]:b[j + 1]]:
                a &= x
            if a != vals[j]:
                ok = False; break
            s += nums[b[j + 1] - 1]
        if ok:
            best = min(best, s)
    return -1 if best == INF else best

for a, v, exp in examples:
    print('brute:', brute(a, v), '| kỳ vọng', exp)

brute: 12 | kỳ vọng 12
brute: 17 | kỳ vọng 17
brute: -1 | kỳ vọng -1


**Độ phức tạp:** `O(C(n − 1, m − 1) · n)`.

## 6. Ý tưởng thuật toán / bước đột phá

```
groups[r] = các (v, Lmin, Lmax) của AND(l..r)
prev[i] = chi phí tốt nhất kết thúc đoạn trước tại i  (tầng "−1": chỉ prev[−1] = 0)
for j in 0..m-1:
    cur[r] = nums[r] + min(prev[l − 1] for l ∈ [L, R])  với (andValues[j], L, R) ∈ groups[r]
    prev = cur
answer = prev[n − 1]
```

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DFS có nhớ `(i, j, AND hiện tại)`

Tại vị trí `i`, đang xây đoạn `j` với AND tích luỹ `cur`: hoặc kéo dài đoạn, hoặc (nếu `cur & nums[i] == andValues[j]`) kết thúc đoạn tại `i`. Số giá trị `cur` khác nhau nhỏ nên số trạng thái `O(n · m · 17)`.

In [4]:
def memo_dfs(nums, vals):
    n, m = len(nums), len(vals)
    sys.setrecursionlimit(max(10000, 4 * n + 100))
    @lru_cache(maxsize=None)
    def f(i, j, cur):
        if j == m:
            return 0 if i == n else INF
        if i == n:
            return INF
        c = cur & nums[i]
        if c & vals[j] != vals[j]:                 # đã mất bit cần có: không bao giờ khớp được nữa
            return INF
        best = f(i + 1, j, c)                      # kéo dài đoạn j
        if c == vals[j]:
            best = min(best, nums[i] + f(i + 1, j + 1, -1))   # kết thúc đoạn j tại i
        return best
    r = f(0, 0, -1); f.cache_clear()
    return -1 if r == INF else r

for a, v, exp in examples:
    print('DFS có nhớ:', memo_dfs(a, v), '| kỳ vọng', exp)

DFS có nhớ: 12 | kỳ vọng 12
DFS có nhớ: 17 | kỳ vọng 17
DFS có nhớ: -1 | kỳ vọng -1


### 7.2. Cách 3 — DP theo tầng `O(m · n²)` (trực tiếp)

In [5]:
def layered_n2(nums, vals):
    n, m = len(nums), len(vals)
    prev = [INF] * n                                 # prev[i]: đoạn trước kết thúc tại i
    for j in range(m):
        cur = [INF] * n
        for r in range(n):
            a = -1
            for l in range(r, -1, -1):
                a &= nums[l]
                if a < vals[j]:
                    break
                if a == vals[j]:
                    before = 0 if (j == 0 and l == 0) else (prev[l - 1] if l >= 1 and j > 0 else INF)
                    if before + nums[r] < cur[r]:
                        cur[r] = before + nums[r]
        prev = cur
    return -1 if prev[n - 1] == INF else prev[n - 1]

for a, v, exp in examples:
    print('theo tầng O(m·n²):', layered_n2(a, v), '| kỳ vọng', exp)

theo tầng O(m·n²): 12 | kỳ vọng 12
theo tầng O(m·n²): 17 | kỳ vọng 17
theo tầng O(m·n²): -1 | kỳ vọng -1


### 7.3. Cách 4 — Nhóm AND + sparse table `O(m · n log n)` (lời giải chính)

In [6]:
def groups_sparse(nums, vals):
    n, m = len(nums), len(vals); G = groups_by_r(nums)
    prev = None
    for j in range(m):
        # bảng B[t] = chi phí tốt nhất để đoạn j bắt đầu tại l = t  (= prev[t − 1], hoặc 0 nếu j = 0 và t = 0)
        if j == 0:
            B = [0] + [INF] * (n - 1)
        else:
            B = [INF] + prev[:n - 1]
        sp = [B]; k = 1
        while (1 << k) <= n:
            p = sp[-1]; h = 1 << (k - 1)
            sp.append([min(p[i], p[i + h]) for i in range(n - (1 << k) + 1)]); k += 1
        def qmin(L, R):
            t = (R - L + 1).bit_length() - 1
            return min(sp[t][L], sp[t][R - (1 << t) + 1])
        cur = [INF] * n; target = vals[j]
        for r in range(n):
            for v, lo, hi in G[r]:
                if v == target:
                    best = qmin(lo, hi)
                    if best < INF:
                        cur[r] = best + nums[r]
                    break
                if v < target:                       # các nhóm sau còn nhỏ hơn
                    break
        prev = cur
    return -1 if prev[n - 1] == INF else prev[n - 1]

for a, v, exp in examples:
    print('nhóm AND + sparse table:', groups_sparse(a, v), '| kỳ vọng', exp)

nhóm AND + sparse table: 12 | kỳ vọng 12
nhóm AND + sparse table: 17 | kỳ vọng 17
nhóm AND + sparse table: -1 | kỳ vọng -1


(Nhóm trong `G[r]` được xếp theo `l` giảm dần ⇒ giá trị AND **giảm dần** ⇒ có thể dừng sớm khi `v < target`.)

## 8. TRACE từng bước

In bảng `dp` từng tầng cho ví dụ 2.

In [7]:
a, vals, _ = examples[1]; n = len(a); G = groups_by_r(a); prev = None
for j, t in enumerate(vals):
    B = [0] + [INF] * (n - 1) if j == 0 else [INF] + prev[:n - 1]
    cur = [INF] * n
    for r in range(n):
        for v, lo, hi in G[r]:
            if v == t:
                b = min(B[lo:hi + 1]); cur[r] = b + a[r] if b < INF else INF; break
    print(f'tầng {j} (AND = {t}): dp = {["∞" if x == INF else x for x in cur]}')
    prev = cur
print('đáp án dp[m−1][n−1] =', prev[-1])

tầng 0 (AND = 0): dp = ['∞', '∞', 5, 7, 7, 7, 5]
tầng 1 (AND = 7): dp = ['∞', '∞', '∞', 12, 12, 12, '∞']
tầng 2 (AND = 5): dp = ['∞', '∞', '∞', '∞', '∞', '∞', 17]
đáp án dp[m−1][n−1] = 17


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def minimumValueSum(self, nums: List[int], andValues: List[int]) -> int:
        n, m = len(nums), len(andValues)
        INF = float('inf')
        G = []
        cur = []
        for r, x in enumerate(nums):
            nxt = [(x, r, r)]
            for v, lo, hi in cur:
                w = v & x
                if w == nxt[-1][0]:
                    nxt[-1] = (w, lo, nxt[-1][2])
                else:
                    nxt.append((w, lo, hi))
            cur = nxt
            G.append(cur)
        prev = None
        for j in range(m):
            B = [0] + [INF] * (n - 1) if j == 0 else [INF] + prev[:n - 1]
            sp = [B]
            k = 1
            while (1 << k) <= n:
                p = sp[-1]
                h = 1 << (k - 1)
                sp.append([p[i] if p[i] < p[i + h] else p[i + h] for i in range(n - (1 << k) + 1)])
                k += 1
            target = andValues[j]
            res = [INF] * n
            for r in range(n):
                for v, lo, hi in G[r]:
                    if v == target:
                        t = (hi - lo + 1).bit_length() - 1
                        a, b = sp[t][lo], sp[t][hi - (1 << t) + 1]
                        best = a if a < b else b
                        if best < INF:
                            res[r] = best + nums[r]
                        break
                    if v < target:
                        break
            prev = res
        return -1 if prev[n - 1] == INF else prev[n - 1]

for a, v, exp in examples:
    got = Solution().minimumValueSum(a, v)
    print(got, 'OK' if got == exp else 'SAI')

12 OK
17 OK
-1 OK


## 10. Vì sao thuật toán đúng

1. **DP:** một cách chia hợp lệ ⇔ dãy đầu phải `r_0 < r_1 < … < r_{m−1} = n − 1` với `AND(r_{j−1} + 1 .. r_j) = andValues[j]`; tổng giá trị `Σ nums[r_j]`. `dp_j[r]` tối ưu từng tầng (cấu trúc con tối ưu: tiền tố của một cách chia tối ưu là tối ưu cho tiền tố).
2. **Nhóm AND:** với `r` cố định, `AND(l..r)` không tăng khi `l` giảm ⇒ tập `{l : AND(l..r) = v}` là một đoạn — nhóm `(v, Lmin, Lmax)` mô tả đúng đoạn đó (gộp các `l` liền nhau cùng giá trị).
3. **Min đoạn:** `dp_j[r] = nums[r] + min_{l ∈ [L, R]} B_j[l]` với `B_j[l] = dp_{j−1}[l − 1]` (hoặc `0` khi `j = 0, l = 0`) — sparse table trả lời đúng min.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Brute | `O(C(n−1, m−1) · n)` | `O(n)` |
| 2. DFS có nhớ | `O(n · m · log V)` trạng thái | như vậy (+ đệ quy sâu `n`) |
| 3. Theo tầng trực tiếp | `O(m · n²)` (dừng sớm) | `O(n)` |
| 4. Nhóm + sparse table | `O(n log V + m · n log n)` | `O(n log n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'memo': memo_dfs, 'layered': layered_n2, 'sparse': groups_sparse,
         'submit': lambda a, v: Solution().minimumValueSum(a, v)}
edge = [([5], [5], 5), ([5], [4], -1), ([3, 3, 3], [3], 3), ([3, 3, 3], [3, 3, 3], 9), ([3, 3, 3], [3, 3], 6),
        ([1, 2], [0], 2), ([6, 5, 4], [4, 4], 9), ([7, 6, 6, 5], [6, 4], 11)]
for a, v, exp in examples + edge:
    assert brute(a, v) == exp, (a, v, brute(a, v))
    for name, f in impls.items():
        assert f(a, v) == exp, (name, a, v)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Giải thích: `[3, 3, 3]` với `[3, 3]`: cắt `[3] [3, 3]` hoặc `[3, 3] [3]` đều có tổng `6`. `[7, 6, 6, 5]` với `[6, 4]`: `[7, 6] [6, 5]` ⇒ `6 + 5 = 11`.

### 12.2. Đối chiếu ngẫu nhiên với brute

In [10]:
rng = random.Random(3117)
def rand_case(n, m, V):
    nums = [rng.randint(1, V) for _ in range(n)]
    if rng.random() < 0.6:                           # sinh bộ khả thi
        cuts = sorted(rng.sample(range(1, n), m - 1)); b = [0] + cuts + [n]; vals = []
        for j in range(m):
            a = -1
            for x in nums[b[j]:b[j + 1]]:
                a &= x
            vals.append(a)
    else:
        vals = [rng.randint(0, V) for _ in range(m)]
    return nums, vals

cnt = 0
for _ in range(400):
    n = rng.randint(1, 9); m = rng.randint(1, min(n, 4))
    a, v = rand_case(n, m, rng.choice([3, 7, 15, 31]))
    exp = brute(a, v)
    for name, f in impls.items():
        assert f(a, v) == exp, (name, a, v)
    cnt += 1
print('Khớp brute trên', cnt, 'bộ ngẫu nhiên.')

Khớp brute trên 400 bộ ngẫu nhiên.


### 12.3. Ca lớn `n = 10⁴`, `m = 10`

In [11]:
N = 10 ** 4
cases = {}
a, v = rand_case(N, 10, 2 ** 17 - 1); cases['ngẫu nhiên khả thi'] = (a, v)
a = [rng.choice([2 ** 17 - 1, 2 ** 16 - 1]) for _ in range(N)]; cases['ít giá trị'] = (a, [2 ** 16 - 1] * 10)
cases['toàn bằng nhau'] = ([12345] * N, [12345] * 10)
for name, (a, v) in cases.items():
    t = time.time(); x = Solution().minimumValueSum(a, v); t1 = time.time() - t
    print(f'{name:20s}: {x:8d} | nộp {t1:.2f}s')
# đối chiếu với DP tầng trực tiếp O(m·n²) trên cỡ vừa (n = 1500) để giữ thời gian chạy notebook ngắn
for _ in range(3):
    a, v = rand_case(1500, 10, 2 ** 17 - 1)
    assert Solution().minimumValueSum(a, v) == layered_n2(a, v)
print('Cỡ vừa n = 1500, m = 10: khớp DP tầng trực tiếp.')
assert Solution().minimumValueSum([12345] * N, [12345] * 10) == 12345 * 10
print('Toàn bằng nhau: 10 đoạn, mỗi đoạn giá trị 12345: đúng.')

ngẫu nhiên khả thi  :   113699 | nộp 0.10s


ít giá trị          :   655350 | nộp 0.08s


toàn bằng nhau      :   123450 | nộp 0.07s


Cỡ vừa n = 1500, m = 10: khớp DP tầng trực tiếp.
Toàn bằng nhau: 10 đoạn, mỗi đoạn giá trị 12345: đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Lấy phần tử **đầu** đoạn làm giá trị

In [12]:
def wrong_first_value(nums, vals):
    n, m = len(nums), len(vals); best = INF
    for cuts in itertools.combinations(range(1, n), m - 1):
        b = [0] + list(cuts) + [n]; s = 0; ok = True
        for j in range(m):
            a = -1
            for x in nums[b[j]:b[j + 1]]:
                a &= x
            if a != vals[j]:
                ok = False; break
            s += nums[b[j]]                                                   # SAI: phải là phần tử cuối
        if ok:
            best = min(best, s)
    return -1 if best == INF else best

print('sai :', wrong_first_value(*examples[1][:2]), '| đúng:', Solution().minimumValueSum(*examples[1][:2]))

sai : 14 | đúng: 17


**Vì sao sai:** đề định nghĩa giá trị của mảng là phần tử **cuối**.

### Lỗi 2 — Tham lam cắt sớm nhất (đoạn cuối lấy phần còn lại)

In [13]:
def wrong_greedy(nums, vals):
    n = len(nums); i = 0; s = 0
    for j, v in enumerate(vals):
        a = -1
        if j == len(vals) - 1:                                                # đoạn cuối: phần còn lại
            for x in nums[i:]:
                a &= x
            return s + nums[-1] if i < n and a == v else -1
        found = False
        while i < n:
            a &= nums[i]; i += 1
            if a == v:
                found = True; break                                            # SAI: cắt ngay khi khớp
        if not found:
            return -1
        s += nums[i - 1]
    return -1

diff = None
for _ in range(5000):
    n = rng.randint(2, 7); m = rng.randint(2, min(n, 3)); a, v = rand_case(n, m, 7)
    if wrong_greedy(a, v) != Solution().minimumValueSum(a, v):
        diff = (a, v); break
print('nums, andValues =', diff, '| sai :', wrong_greedy(*diff), '| đúng:', Solution().minimumValueSum(*diff))

nums, andValues = ([7, 3, 1, 6, 4, 3, 5], [7, 0, 1]) | sai : -1 | đúng: 16


**Vì sao sai:** cắt sớm có thể làm đoạn sau không khớp (hoặc đắt hơn); kéo dài đoạn hiện tại (AND vẫn bằng giá trị cần) đôi khi mới đúng. Cần DP.

### Lỗi 3 — Chấp nhận AND "chứa đủ bit" (`AND & v == v`) thay vì bằng đúng

In [14]:
def wrong_superset(nums, vals):
    n, m = len(nums), len(vals); best = INF
    for cuts in itertools.combinations(range(1, n), m - 1):
        b = [0] + list(cuts) + [n]; s = 0; ok = True
        for j in range(m):
            a = -1
            for x in nums[b[j]:b[j + 1]]:
                a &= x
            if a & vals[j] != vals[j]:                                         # SAI
                ok = False; break
            s += nums[b[j + 1] - 1]
        if ok:
            best = min(best, s)
    return -1 if best == INF else best

args = ([3, 5, 5, 4], [3, 5, 0])
print('sai :', wrong_superset(*args), '| đúng:', Solution().minimumValueSum(*args))

sai : 12 | đúng: -1


**Vì sao sai:** cắt tỉa "đã mất bit thì dừng" (dùng ở cách 2) **không** thay cho kiểm tra bằng nhau; `andValues[j] = 0` thì mọi đoạn đều "chứa đủ bit".

### Lỗi 4 — Quên rằng đoạn đầu phải bắt đầu tại chỉ số 0 (cho `dp` tầng 0 mọi `l` đều bằng 0)

In [15]:
def wrong_first_layer(nums, vals):
    n, m = len(nums), len(vals); G = groups_by_r(nums); prev = None
    for j in range(m):
        B = [0] * n if j == 0 else [INF] + prev[:n - 1]                      # SAI: đoạn đầu phải có l = 0
        cur = [INF] * n
        for r in range(n):
            for v, lo, hi in G[r]:
                if v == vals[j]:
                    b = min(B[lo:hi + 1]); cur[r] = b + nums[r] if b < INF else INF; break
        prev = cur
    return -1 if prev[n - 1] == INF else prev[n - 1]

args = ([1, 2], [2])
print('sai :', wrong_first_layer(*args), '| đúng:', Solution().minimumValueSum(*args))

sai : 2 | đúng: -1


**Vì sao sai:** cho phép đoạn đầu bắt đầu ở giữa mảng ⇒ bỏ qua các phần tử đầu không thuộc đoạn nào. Với `nums = [1, 2]`, `andValues = [2]`: đoạn `[2]` (chỉ `nums[1]`) có AND `2` nhưng `nums[0]` bị bỏ sót — đáp án đúng là `-1` vì AND cả mảng là `0`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chia mảng thành `m` đoạn, điều kiện trên AND/OR/GCD từng đoạn" ⇒ DP tầng + nhóm giá trị theo đầu phải (`≤ log V` nhóm).
- Chuyển trạng thái là min trên một **đoạn** đầu trái ⇒ sparse table / deque.
- `m` nhỏ (`≤ 10`) cho phép DP tầng `O(m · n · …)`.

## 15. Biến thể của bài

1. Giá trị đoạn là **tổng** đoạn ⇒ cùng khung, cộng tiền tố.
2. Điều kiện **OR** của đoạn bằng `orValues[j]` ⇒ OR không giảm khi `l` giảm — vẫn là đoạn.
3. Không cố định `m`: chia tuỳ ý thành các đoạn có AND `= k` ⇒ một tầng duy nhất.
4. Tối đa thay vì tối thiểu ⇒ sparse table max.

In [16]:
# Biến thể 2: điều kiện OR
def or_version(nums, vals):
    n, m = len(nums), len(vals)
    G = []; cur = []
    for r, x in enumerate(nums):
        nxt = [(x, r, r)]
        for v, lo, hi in cur:
            w = v | x
            if w == nxt[-1][0]: nxt[-1] = (w, lo, nxt[-1][2])
            else: nxt.append((w, lo, hi))
        cur = nxt; G.append(cur)
    prev = None
    for j in range(m):
        B = [0] + [INF] * (n - 1) if j == 0 else [INF] + prev[:n - 1]
        res = [INF] * n
        for r in range(n):
            for v, lo, hi in G[r]:
                if v == vals[j]:
                    b = min(B[lo:hi + 1]); res[r] = b + nums[r] if b < INF else INF; break
        prev = res
    return -1 if prev[n - 1] == INF else prev[n - 1]

def or_brute(nums, vals):
    n, m = len(nums), len(vals); best = INF
    for cuts in itertools.combinations(range(1, n), m - 1):
        b = [0] + list(cuts) + [n]; s = 0; ok = True
        for j in range(m):
            o = 0
            for x in nums[b[j]:b[j + 1]]: o |= x
            if o != vals[j]: ok = False; break
            s += nums[b[j + 1] - 1]
        if ok: best = min(best, s)
    return -1 if best == INF else best

for _ in range(300):
    n = rng.randint(1, 8); m = rng.randint(1, min(n, 3)); a = [rng.randint(1, 7) for _ in range(n)]
    cuts = sorted(rng.sample(range(1, n), m - 1)); b = [0] + cuts + [n]
    vals = [__import__('functools').reduce(lambda p, q: p | q, a[b[j]:b[j + 1]]) for j in range(m)]
    assert or_version(a, vals) == or_brute(a, vals)
print('Biến thể OR: khớp brute.')

Biến thể OR: khớp brute.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Thử mọi cách cắt | tổ hợp | Oracle |
| 2 | DFS có nhớ | `O(n m log V)` | Ngắn, đệ quy sâu |
| 3 | Tầng trực tiếp | `O(m n²)` | Dễ hiểu |
| 4 | Nhóm AND + sparse table | `O(m n log n)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [4, 4, 4]`, `andValues = [4, 4]`.

**Bài 2 (tính tay):** `nums = [7, 3, 1, 1]`, `andValues = [3, 1]`.

**Bài 3 (điền TODO):** trả về luôn **vị trí các vết cắt** của một cách chia tối ưu.

**Bài 4 (tự viết lại):** biến thể 3 — chia toàn bộ mảng thành **nhiều nhất** bao nhiêu đoạn mà mỗi đoạn có AND `= k` (hoặc `-1`).

**Bài 5 (mini challenge):** biến thể 4 — tổng giá trị **lớn nhất**.

In [17]:
def cuts_todo(nums, vals):
    # TODO
    raise NotImplementedError

def max_segments_todo(nums, k):
    # TODO
    raise NotImplementedError

def maximize_todo(nums, vals):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Hai đoạn, mỗi đoạn AND 4. (2) Giá trị luôn là 4. (3) `8`.

**Bài 2.** (1) Đoạn đầu AND = 3: `[7, 3]`. (2) Đoạn sau `[1, 1]` AND 1. (3) `3 + 1 = 4`.

**Bài 3.** (1) Lưu `argmin` cùng sparse table (cặp `(giá trị, chỉ số)`). (2) Truy ngược từ `n − 1`. (3) Kiểm AND từng đoạn.

**Bài 4.** (1) Một tầng: `best[r]` = số đoạn tối đa chia `nums[0..r]`. (2) `best[r] = 1 + max best[l − 1]` trên khoảng `l` có `AND(l..r) = k`. (3) Hoặc tham lam: cắt khi AND về đúng `k` rồi… — cẩn thận, kiểm bằng brute.

**Bài 5.** (1) Thay `min` bằng `max`, `INF` bằng `−INF`. (2) Sparse table max. (3) Kiểm brute.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 3
assert Solution().minimumValueSum([4, 4, 4], [4, 4]) == 8
assert Solution().minimumValueSum([7, 3, 1, 1], [3, 1]) == 4

def cuts(nums, vals):
    n, m = len(nums), len(vals); G = groups_by_r(nums); layers = []; prev = None
    for j in range(m):
        B = [(0, -1)] + [(INF, -1)] * (n - 1) if j == 0 else [(INF, -1)] + [(prev[t][0], t) for t in range(n - 1)]
        cur = [(INF, -1)] * n
        for r in range(n):
            for v, lo, hi in G[r]:
                if v == vals[j]:
                    b = min(B[lo:hi + 1])
                    if b[0] < INF: cur[r] = (b[0] + nums[r], b[1])
                    break
        layers.append(cur); prev = cur
    if prev[n - 1][0] == INF:
        return None
    ends = []; r = n - 1
    for j in range(m - 1, -1, -1):
        ends.append(r); r = layers[j][r][1]
    return ends[::-1]                                     # chỉ số kết thúc mỗi đoạn

for _ in range(300):
    n = rng.randint(1, 9); m = rng.randint(1, min(n, 4)); a, v = rand_case(n, m, 15)
    e = cuts(a, v); best = brute(a, v)
    if e is None:
        assert best == -1
    else:
        b = [0] + [x + 1 for x in e]
        assert e[-1] == n - 1 and sum(a[x] for x in e) == best
        for j in range(m):
            acc = -1
            for x in a[b[j]:b[j + 1]]: acc &= x
            assert acc == v[j]
print('Bài 1: 8 | Bài 2: 4 | Bài 3: vết cắt hợp lệ và tối ưu; ví dụ 2 ->', cuts(*examples[1][:2]))

Bài 1: 8 | Bài 2: 4 | Bài 3: vết cắt hợp lệ và tối ưu; ví dụ 2 -> [2, 3, 6]


In [19]:
# Bài 4, 5
def max_segments(nums, k):
    n = len(nums); G = groups_by_r(nums); best = [-INF] * n
    for r in range(n):
        for v, lo, hi in G[r]:
            if v == k:
                cand = max((best[l - 1] if l >= 1 else 0) for l in range(lo, hi + 1))
                if cand > -INF: best[r] = cand + 1
                break
    return best[n - 1] if best[n - 1] > -INF else -1

def maximize(nums, vals):
    n, m = len(nums), len(vals); G = groups_by_r(nums); prev = None
    for j in range(m):
        B = [0] + [-INF] * (n - 1) if j == 0 else [-INF] + prev[:n - 1]
        cur = [-INF] * n
        for r in range(n):
            for v, lo, hi in G[r]:
                if v == vals[j]:
                    b = max(B[lo:hi + 1]); cur[r] = b + nums[r] if b > -INF else -INF; break
        prev = cur
    return -1 if prev[n - 1] == -INF else prev[n - 1]

def brute_max_seg(nums, k):
    n = len(nums); best = -1
    for mask in range(1 << (n - 1)):
        b = [0] + [i + 1 for i in range(n - 1) if mask >> i & 1] + [n]; ok = True
        for j in range(len(b) - 1):
            acc = -1
            for x in nums[b[j]:b[j + 1]]: acc &= x
            if acc != k: ok = False; break
        if ok: best = max(best, len(b) - 1)
    return best

def brute_max(nums, vals):
    n, m = len(nums), len(vals); best = -1
    for c in itertools.combinations(range(1, n), m - 1):
        b = [0] + list(c) + [n]; ok = True; s = 0
        for j in range(m):
            acc = -1
            for x in nums[b[j]:b[j + 1]]: acc &= x
            if acc != vals[j]: ok = False; break
            s += nums[b[j + 1] - 1]
        if ok: best = max(best, s)
    return best

for _ in range(300):
    n = rng.randint(1, 9); a = [rng.randint(1, 7) for _ in range(n)]; k = rng.choice(a + [0, 1])
    assert max_segments(a, k) == brute_max_seg(a, k)
    m = rng.randint(1, min(n, 3)); a2, v2 = rand_case(n, m, 7)
    assert maximize(a2, v2) == brute_max(a2, v2)
print('Bài 4 (nhiều đoạn nhất) và Bài 5 (tổng lớn nhất) khớp brute.')

Bài 4 (nhiều đoạn nhất) và Bài 5 (tổng lớn nhất) khớp brute.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
G[r] = nhóm (AND, Lmin, Lmax) của các mảng con kết thúc tại r
for j: B[l] = dp_{j-1}[l-1] (B[0] = 0 nếu j = 0); sparse table min trên B
       dp_j[r] = nums[r] + min(B[Lmin..Lmax]) với nhóm có AND = andValues[j]
answer = dp_{m-1}[n-1] hoặc -1
```

**Checklist:**

- [ ] Giá trị đoạn là phần tử **cuối**?
- [ ] AND **bằng đúng** `andValues[j]`?
- [ ] Đoạn đầu bắt đầu ở 0, đoạn cuối kết thúc ở `n − 1`?
- [ ] Trả `-1` khi không chia được?